# Setup

In [2]:
import sys
import os
import numpy as np
import pandas as pd
import random
from matplotlib import pyplot as plt
import tensorflow as tf
from numpy.typing import NDArray

RANDOM_SEED = 42

sys.path.append(os.path.abspath("."))
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

2025-08-31 23:16:41.440070: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-08-31 23:16:41.440118: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-08-31 23:16:41.440134: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-08-31 23:16:41.443803: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
(X_train_full, y_train_full), (X_test, y_test) = tf.keras.datasets.cifar10.load_data()

VALID_SIZE = 10000

X_valid = X_train_full[:VALID_SIZE]
y_valid = y_train_full[:VALID_SIZE]

X_train = X_train_full[VALID_SIZE:]
y_train = y_train_full[VALID_SIZE:]

In [4]:
X_train.shape

(40000, 32, 32, 3)

In [5]:
X_test.shape

(10000, 32, 32, 3)

In [6]:
# Use smaller subsets while searching to make the whole process faster.
X_train_s = X_train[:10000]
y_train_s = y_train[:10000]

- DNN with `20` hidden layers + `100` neurons on each layer
- `He` initialisation + `Swish` activation funcion
- `Nadam` optimisation
- early stopping.

In [7]:
tf.keras.backend.clear_session()

def create_model():
    model = tf.keras.Sequential([
        tf.keras.layers.Flatten(input_shape=[32, 32, 3])
    ])

    for _ in range(20):

        # He initialisation + Swish activation function
        layer = tf.keras.layers.Dense(
            100,
            activation="swish",
            kernel_initializer="he_normal"
        )

        model.add(layer)

    # output layer
    model.add(tf.keras.layers.Dense(10, activation="softmax"))  # according to the dataset, there's 10 possible values. Hence 10 + softmax

    return model

In [53]:
create_model().summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 flatten (Flatten)           (None, 3072)              0         
                                                                 
 dense (Dense)               (None, 100)               307300    
                                                                 
 dense_1 (Dense)             (None, 100)               10100     
                                                                 
 dense_2 (Dense)             (None, 100)               10100     
                                                                 
 dense_3 (Dense)             (None, 100)               10100     
                                                                 
 dense_4 (Dense)             (None, 100)               10100     
                                                                 
 dense_5 (Dense)             (None, 100)               1

I went through the same values manually here from the official solution:

- `1e-5`
- `3e-5`
- `1e-4`
- `3e-4`
- `1e-3`
- `3e-3`
- `1e-2`

Though I personally guess if I had to do this from scratch, I'd have tried to figure it out using one of the Learning Rate Schedules, probably the recommended default one (Performance Scheduling) initially.

Also, it kind of goes against the idea of starting with a large learning rate and decreasing it :D 

In [8]:
from pathlib import Path

earlystop_cb = tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)
checkpoint_cb = tf.keras.callbacks.ModelCheckpoint(
    "cifar10_model.keras",
    monitor='val_accuracy',
    mode='max',
    #save_best_only=True  # <- (kaput for the same reason as in 10_mlp.ipynb)
)

In [ ]:
for lr in [1e-5, 3e-5, 1e-4, 3e-4, 1e-3, 3e-3, 1e-2]:
    run_logdir = Path() / "cifar10" / f"run_{lr:.0e}"

    tf.keras.backend.clear_session()

    model = create_model()
    model.compile(
        loss="sparse_categorical_crossentropy",  # <- we are still dealing with sparse categorical data, just like the MNIST dataset
        optimizer=tf.keras.optimizers.Nadam(learning_rate=lr),  # Adam + Nesterov trick
        metrics=["accuracy"]
    )

    tensorboard_cb = tf.keras.callbacks.TensorBoard(run_logdir)

    history = model.fit(
        X_train,
        y_train,
        epochs=10,
        validation_data=(X_valid, y_valid),
        callbacks=[
            earlystop_cb,
            tensorboard_cb,
            checkpoint_cb
        ]
    )

Epoch 1/10
1250/1250 [==============================] - 21s 13ms/step - loss: 15.6705 - accuracy: 0.1131 - val_loss: 4.0413 - val_accuracy: 0.1116
Epoch 2/10
1250/1250 [==============================] - 19s 15ms/step - loss: 3.1792 - accuracy: 0.1310 - val_loss: 2.7516 - val_accuracy: 0.1267
Epoch 3/10
1250/1250 [==============================] - 17s 13ms/step - loss: 2.5284 - accuracy: 0.1407 - val_loss: 2.3946 - val_accuracy: 0.1437
Epoch 4/10
1250/1250 [==============================] - 19s 15ms/step - loss: 2.2966 - accuracy: 0.1757 - val_loss: 2.2139 - val_accuracy: 0.1927
Epoch 5/10
1250/1250 [==============================] - 20s 16ms/step - loss: 2.1657 - accuracy: 0.2067 - val_loss: 2.1262 - val_accuracy: 0.2148
Epoch 6/10
1250/1250 [==============================] - 17s 13ms/step - loss: 2.0771 - accuracy: 0.2352 - val_loss: 2.0565 - val_accuracy: 0.2418
Epoch 7/10
1250/1250 [==============================] - 18s 14ms/step - loss: 2.0105 - accuracy: 0.2591 - val_loss: 1.9911 

Here were my TensorBoard reults for the proposed values:

![My Tensorflow Results](./2025-08-31_22-24.png)

Again I think per the suggestions the idea would've been to actually start at high values and just use one of the strategies mentioned on p.389 (Performance scheduling being the default suggested strategy) but I decided to recreate the setup that Geron discussed him using to see what he might've seen and how the optimal selection would've looked.

In [ ]:
# So, Geron mentions he ended up testing 5e-5 too and it turning out slightly better. Let's see that then
tf.keras.backend.clear_session()

model = create_model()
model.compile(
    loss="sparse_categorical_crossentropy",  # <- we are still dealing with sparse categorical data, just like the MNIST dataset
    optimizer=tf.keras.optimizers.Nadam(learning_rate=5e-5),  # Adam + Nesterov trick
    metrics=["accuracy"]
)

tensorboard_cb = tf.keras.callbacks.TensorBoard(Path() / "cifar10" / f"run_5e-5")

history = model.fit(
    X_train,
    y_train,
    epochs=10,
    validation_data=(X_valid, y_valid),
    callbacks=[
        earlystop_cb,
        tensorboard_cb,
        checkpoint_cb
    ]
)

Epoch 1/10
1250/1250 [==============================] - 19s 12ms/step - loss: 9.0002 - accuracy: 0.1525 - val_loss: 2.2011 - val_accuracy: 0.1992
Epoch 2/10
1250/1250 [==============================] - 19s 15ms/step - loss: 2.1151 - accuracy: 0.2267 - val_loss: 2.0733 - val_accuracy: 0.2363
Epoch 3/10
1250/1250 [==============================] - 15s 12ms/step - loss: 2.0022 - accuracy: 0.2639 - val_loss: 1.9703 - val_accuracy: 0.2861
Epoch 4/10
1250/1250 [==============================] - 19s 15ms/step - loss: 1.9266 - accuracy: 0.2919 - val_loss: 1.8663 - val_accuracy: 0.3122
Epoch 5/10
1250/1250 [==============================] - 15s 12ms/step - loss: 1.8750 - accuracy: 0.3151 - val_loss: 1.8625 - val_accuracy: 0.3193
Epoch 6/10
1250/1250 [==============================] - 19s 15ms/step - loss: 1.8246 - accuracy: 0.3350 - val_loss: 1.8083 - val_accuracy: 0.3487
Epoch 7/10
1250/1250 [==============================] - 15s 12ms/step - loss: 1.7804 - accuracy: 0.3555 - val_loss: 1.7491 -

![My results of 5e-5](./2025-08-31_22-33.png)

So let's go with `5e-5` for now, even though in my case I think in my case `1e-4` is the better option.

In [9]:
tf.keras.backend.clear_session()

model = create_model()
model.compile(
    loss="sparse_categorical_crossentropy",  # <- we are still dealing with sparse categorical data, just like the MNIST dataset
    optimizer=tf.keras.optimizers.Nadam(learning_rate=5e-5),  # Adam + Nesterov trick
    metrics=["accuracy"]
)

earlystop_cb = tf.keras.callbacks.EarlyStopping(
    patience=20, # <- make it wait a bit longer in the full run
    restore_best_weights=True
)
tensorboard_cb = tf.keras.callbacks.TensorBoard(Path() / "cifar10" / f"full_run_5e-5")

# fit for 100 epochs!
history = model.fit(
    X_train,
    y_train,
    epochs=100, # <- full run this time, not just 10 epochs as during the learning rate finding...
    validation_data=(X_valid, y_valid),
    callbacks=[
        earlystop_cb,
        tensorboard_cb,
        checkpoint_cb
    ]
)

2025-08-31 23:17:11.073804: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-08-31 23:17:11.080101: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-08-31 23:17:11.080125: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-08-31 23:17:11.081468: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-08-31 23:17:11.081503: I tensorflow/compile

Epoch 1/100


2025-08-31 23:17:13.795492: I tensorflow/compiler/xla/service/service.cc:168] XLA service 0x745cc1e06840 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-08-31 23:17:13.795522: I tensorflow/compiler/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA GeForce RTX 4080 Laptop GPU, Compute Capability 8.9
2025-08-31 23:17:13.801806: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-08-31 23:17:13.809153: I tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:442] Loaded cuDNN version 8700
2025-08-31 23:17:13.868148: I ./tensorflow/compiler/jit/device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1250/1250 [==============================] - 21s 14ms/step - loss: 6.6086 - accuracy: 0.1543 - val_loss: 2.2072 - val_accuracy: 0.1988
Epoch 2/100
1250/1250 [==============================] - 16s 13ms/step - loss: 2.1276 - accuracy: 0.2188 - val_loss: 2.0919 - val_accuracy: 0.2321
Epoch 3/100
1250/1250 [==============================] - 17s 14ms/step - loss: 2.0183 - accuracy: 0.2566 - val_loss: 1.9827 - val_accuracy: 0.2682
Epoch 4/100
1250/1250 [==============================] - 15s 12ms/step - loss: 1.9324 - accuracy: 0.2889 - val_loss: 1.8801 - val_accuracy: 0.3089
Epoch 5/100
1250/1250 [==============================] - 19s 15ms/step - loss: 1.8636 - accuracy: 0.3176 - val_loss: 1.8242 - val_accuracy: 0.3313
Epoch 6/100
1250/1250 [==============================] - 15s 12ms/step - loss: 1.8040 - accuracy: 0.3434 - val_loss: 1.7839 - val_accuracy: 0.3403
Epoch 7/100
1250/1250 [==============================] - 18s 14ms/step - loss: 1.7558 - accuracy: 0.3630 - val_loss: 1.7166 - val_

Bonus visualisation of remote WSL2 VM, just to see what it's actually up to as it's converging...

```text

Every 2.0s: nvidia-smi                                                                                                                                                                                                                                                                      shane: Sun Aug 31 22:41:49 2025

Sun Aug 31 22:41:49 2025
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.76.07              Driver Version: 581.08         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4080 ...    On  |   00000000:01:00.0 Off |                  N/A |
| N/A   57C    P5             14W /   79W |    9736MiB /  12282MiB |     38%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+------------------------+----------------------+

+-----------------------------------------------------------------------------------------+
| Processes:                                                                              |
|  GPU   GI   CI              PID   Type   Process name                        GPU Memory |
|        ID   ID                                                               Usage      |
|=========================================================================================|
|    0   N/A  N/A          343769      C   /python3.10                           N/A      |
+-----------------------------------------------------------------------------------------+
```

For me the whole processing is surprisingly taking longer than for Geron. 

In [ ]:
model.evaluate(X_valid, y_valid) # [loss, accuracy]

313/313 [==============================] - 1s 3ms/step - loss: 1.4890 - accuracy: 0.4823


[1.4889771938323975, 0.4823000133037567]

More or less similar result, I guess. `48.2%` accuracy. Our best model got saved to `cifar10_model.keras`